### Importing the packages

In [ ]:
!pip3 install openai chromadb python-dotenv

# load the packages

In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
import chromadb

In [ ]:
from google.colab import userdata
openaiapikey=userdata.get('groq_api_key')
print(openaiapikey[:10])

In [ ]:
client=OpenAI(api_key=openaiapikey,
    base_url="https://api.groq.com/openai/v1")
print("groq client created successfully!")

#communicating to the groq api

In [ ]:
#GROQ API call
response = client.responses.create(
    input="what is rag?explain in 3 bullet points?",
    model="openai/gpt-oss-20b",
)
print(response.output_text)

#OPENAI API call
# response=client.chat.completions.create(
#     messages=[
#         {
#             "role": "system",
#             "content": "you are a helpful assistant",
#         },
#         {
#             "role": "user",
#             "content": "what is rag?explain in 3 bullet points?"
#         }
#     ],
# )
# print(response.choices[0].message.content)

In [ ]:
response.model_dump_json()

# Load and chunk the data from documents

In [ ]:
with open("/content/company_hr_policy.txt","r") as f:
  hr_policy=f.read()

with open("/content/engineering_standards.txt","r") as f:
  engineering=f.read()

with open("/content/onboarding_guide.txt","r") as f:
  onboarding=f.read()

with open("/content/product_knowledge_base.txt","r") as f:
  product=f.read()

with open("/content/security_policy.txt","r") as f:
  security=f.read()


In [ ]:
print("chars:",len(hr_policy))
print("words:",len(hr_policy.split()))

# chunking strategy




In [ ]:
def chunking(text,source_name):
  #split on double new lines
  paragraphs=text.strip().split("\n\n")
  chunks=[]
  for para in paragraphs:
    para=para.strip()
    if len(para)==50:
      continue
    if para.startswith("=="):
      continue
    chunks.append({"text":para,"source":source_name})
  return chunks

In [ ]:
hr_chunks=chunking(hr_policy,"HR policy")
engineering=chunking(engineering,"Engineering")
onboarding=chunking(onboarding,"Onboarding")
product=chunking(product,"Product")
security=chunking(security,"Security")

In [ ]:
all_chunks=hr_chunks+engineering+onboarding+product+security
print("total chunks:",len(all_chunks))

In [ ]:
print(len(hr_chunks))

# Embeding


In [ ]:
from sentence_transformers import SentenceTransformer

model=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

sentences='''Session Management:
- Idle sessions timeout after 30 minutes for sensitive systems (AWS, HR portal)
- General sessions (Slack, Jira) timeout after 8 hours
- Always lock your laptop when leaving your desk (Cmd+L on Mac)
- Never leave your laptop unattended in public spaces'''

embeddings=model.encode(sentences)
print(embeddings.shape)
print(embeddings)

# similarity search

In [ ]:
sentence1="i love chicken biryani"
sentence2="i love eating flavoured rice with chicken"
sentence3="i am doing work from home"
embedding1=model.encode(sentence1)
embedding2=model.encode(sentence2)
embedding3=model.encode(sentence3)

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
print(cosine_similarity([embedding1],[embedding2]))
print(cosine_similarity([embedding1],[embedding3]))
print(cosine_similarity([embedding2],[embedding3]))

# storing chunks with Chroma DB

In [ ]:
chroma_client=chromadb.Client()
collection=chroma_client.create_collection(name="Company_docs")

In [ ]:
documents=[]
ids=[]
metadata=[]
for i,chunk in enumerate(all_chunks):
  documents.append(chunk["text"])
  ids.append(f"chunk_{i}")
  metadata.append({"source":chunk["source"]})
print(documents[101])
print(ids[101])
print(metadata[101])

In [ ]:
collection.add(
    documents=documents,
    ids=ids,
    metadatas=metadata
)

#Build retrievel pipeline

In [ ]:
def retrieve(question,n_results=3):
  results=collection.query(
      query_texts=[question],
      n_results=n_results,
  )
  return results['documents'][0],results['metadatas'][0]


In [ ]:
chunks,sources=retrieve("what is the work from home policy?",3)
for i in range(len(chunks)):
  print(f"--- chunks {i+1} ---")
  print(f"Source {sources[i]}")
  print(f"Text {chunks[i]}")

In [ ]:
def ask_rag(question,n_results=3,verbose=True):
  chunks,sources=retrieve(question,n_results)
  if verbose:
    print(f"\n{'='*60}")
    print(f"question: {question}")
    print(f"\n{"="*60}")
    print(f"\nRetrieved {len(chunks)} chunks")
    for i,(chunk,source) in enumerate(zip(chunks,sources)):
      print(f"    [{source['source']}] {chunk[:80]}...")
    print(f"{'-'*60}")

  context="\n\n".join(chunks)
  message=[
      {"role":"system",
      "content":(
          "you are a helpful assistance that answers question based only on the provided context"
          "if the context doesn't contain the information to answer this questions,"
          "say i don't have enough context or information to answer this question"
          "Do not make up information or assume anything. strictly answer only on provided context"
      )
      },{
          "role":"user",
          "content":f"context: {context} \n\n ---\n\n Question:{question}"
      }
  ]


  response = client.responses.create(
      model="openai/gpt-oss-20b",
      input=message,
      # temperature=0.2  #low temperature for actual answers.not for creativity
  )
  if verbose:
    print(f"\n{'='*60}")
    print(f"Answer: {response.output_text} ")
    print(f"\n{'='*60}")
  return response.output_text

print("RAG pipeline is built")

# Test RAG pipeline

In [ ]:
ask_rag("what is work from home policy?")

In [ ]:
ask_rag("how many days of annual leave do employees get?")

In [ ]:
ask_rag("what happens during the probation period?")

In [ ]:
#product kb questions
ask_rag("what are pricing plans for cloud desk pro?")

In [ ]:
ask_rag("how do i cancel my subscription?")

In [ ]:
ask_rag("what is the refund policy?")

In [ ]:
#a question that spans both documents-should say "not enough info"
ask_rag("what is the capital of france?")